In [3]:
import polars as pl

In [37]:
def clean_edges(df: pl.DataFrame) -> pl.DataFrame:
    return (
        df.select(
            [
                "relation",
                "display_relation",
                "x_id",
                "x_type",
                "x_name",
                "x_source",
                "y_id",
                "y_type",
                "y_name",
                "y_source",
            ]
        )
        .drop_nulls()
        .unique()
        .filter(
            ~(
                (pl.col("x_id") == pl.col("y_id"))
                & (pl.col("x_type") == pl.col("y_type"))
                & (pl.col("x_source") == pl.col("y_source"))
                & (pl.col("x_name") == pl.col("y_name"))
            )
        )
    )

In [39]:
protein_names = catalog.load("bronze.gene_names.protein_names")
drug_protein = catalog.load("bronze.drugbank.drug_protein")

[05/27/25 00:49:05] INFO     Loading data from bronze.gene_names.protein_names            ]8;id=934385;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=257867;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\
                             (CSVDataset)...                                                                       

                    INFO     Loading data from bronze.drugbank.drug_protein               ]8;id=346864;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=853185;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\
                             (CSVDataset)...                                                                       

In [40]:
df_protein_drug = (
    drug_protein.join(
        protein_names, left_on="ncbi_gene_id", right_on="ncbi_id", how="left"
    )
    .rename(
        {
            "drug_bank_id": "x_id",
            "ncbi_gene_id": "y_id",
            "drug_bank_name": "x_name",
            "symbol": "y_name",
        }
    )
    .with_columns(
        [
            pl.lit("drug").alias("x_type"),
            pl.lit("DrugBank").alias("x_source"),
            pl.lit("gene").alias("y_type"),
            pl.lit("NCBI").alias("y_source"),
            pl.col("relation").alias("display_relation"),
            pl.lit("drug_protein").alias("relation"),
        ]
    )
)
df_protein_drug

x_id,relation,y_id,uniprot_name,uniprot_id,x_name,y_name,x_type,x_source,y_type,y_source,display_relation
str,str,str,str,str,str,str,str,str,str,str,str
"""DRUGBANK:DB09130""","""drug_protein""","""NCBIGene:2157""","""Coagulation factor VIII""","""UniProtKB:P00451""","""Copper""","""F8""","""drug""","""DrugBank""","""gene""","""NCBI""","""carrier"""
"""DRUGBANK:DB09130""","""drug_protein""","""NCBIGene:2153""","""Coagulation factor V""","""UniProtKB:P12259""","""Copper""","""F5""","""drug""","""DrugBank""","""gene""","""NCBI""","""carrier"""
"""DRUGBANK:DB09140""","""drug_protein""","""NCBIGene:3039""","""Hemoglobin subunit alpha""","""UniProtKB:P69905""","""Oxygen""","""HBA1""","""drug""","""DrugBank""","""gene""","""NCBI""","""carrier"""
"""DRUGBANK:DB09140""","""drug_protein""","""NCBIGene:3040""","""Hemoglobin subunit alpha""","""UniProtKB:P69905""","""Oxygen""","""HBA2""","""drug""","""DrugBank""","""gene""","""NCBI""","""carrier"""
"""DRUGBANK:DB00180""","""drug_protein""","""NCBIGene:866""","""Corticosteroid-binding globuli…","""UniProtKB:P08185""","""Flunisolide""","""SERPINA6""","""drug""","""DrugBank""","""gene""","""NCBI""","""carrier"""
…,…,…,…,…,…,…,…,…,…,…,…
"""DRUGBANK:DB14548""","""drug_protein""","""NCBIGene:201266""","""Zinc transporter ZIP11""","""UniProtKB:Q8N1S5""","""Zinc sulfate, unspecified form""","""SLC39A11""","""drug""","""DrugBank""","""gene""","""NCBI""","""transporter"""
"""DRUGBANK:DB14533""","""drug_protein""","""NCBIGene:55676""","""Zinc transporter 6""","""UniProtKB:Q6NXT4""","""Zinc chloride""","""SLC30A6""","""drug""","""DrugBank""","""gene""","""NCBI""","""transporter"""
"""DRUGBANK:DB14548""","""drug_protein""","""NCBIGene:55676""","""Zinc transporter 6""","""UniProtKB:Q6NXT4""","""Zinc sulfate, unspecified form""","""SLC30A6""","""drug""","""DrugBank""","""gene""","""NCBI""","""transporter"""


In [41]:
clean_edges(df_protein_drug)

relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
str,str,str,str,str,str,str,str,str,str
"""drug_protein""","""target""","""DRUGBANK:DB03461""","""drug""","""Nicotinamide adenine dinucleot…","""DrugBank""","""NCBIGene:1719""","""gene""","""DHFR""","""NCBI"""
"""drug_protein""","""target""","""DRUGBANK:DB00570""","""drug""","""Vinblastine""","""DrugBank""","""NCBIGene:203068""","""gene""","""TUBB""","""NCBI"""
"""drug_protein""","""enzyme""","""DRUGBANK:DB00574""","""drug""","""Fenfluramine""","""DrugBank""","""NCBIGene:1557""","""gene""","""CYP2C19""","""NCBI"""
"""drug_protein""","""target""","""DRUGBANK:DB01592""","""drug""","""Iron""","""DrugBank""","""NCBIGene:54583""","""gene""","""EGLN1""","""NCBI"""
"""drug_protein""","""target""","""DRUGBANK:DB15822""","""drug""","""Pralsetinib""","""DrugBank""","""NCBIGene:4916""","""gene""","""NTRK3""","""NCBI"""
…,…,…,…,…,…,…,…,…,…
"""drug_protein""","""target""","""DRUGBANK:DB06404""","""drug""","""Human C1-esterase inhibitor""","""DrugBank""","""NCBIGene:2161""","""gene""","""F12""","""NCBI"""
"""drug_protein""","""target""","""DRUGBANK:DB12010""","""drug""","""Fostamatinib""","""DrugBank""","""NCBIGene:5156""","""gene""","""PDGFRA""","""NCBI"""
"""drug_protein""","""enzyme""","""DRUGBANK:DB01873""","""drug""","""Epothilone D""","""DrugBank""","""NCBIGene:1576""","""gene""","""CYP3A4""","""NCBI"""


In [42]:
catalog.load("silver.drugbank.drug_drug")

[05/27/25 01:02:10] INFO     Loading data from silver.drugbank.drug_drug (CSVDataset)...  ]8;id=909908;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py\kedro_data_catalog.py]8;;\:]8;id=497983;file:///home/viti/Documents/repos/personal/hardvard/optimuskg/.venv/lib/python3.12/site-packages/kedro/io/kedro_data_catalog.py#757\757]8;;\

relation,display_relation,x_id,x_type,x_name,x_source,y_id,y_type,y_name,y_source
str,str,str,str,str,str,str,str,str,str
"""drug_drug""","""synergistic interaction""","""DRUGBANK:DB00482""","""drug""","""Celecoxib""","""DrugBank""","""DRUGBANK:DB00420""","""drug""","""Promazine""","""DrugBank"""
"""drug_drug""","""synergistic interaction""","""DRUGBANK:DB06470""","""drug""","""Clomethiazole""","""DrugBank""","""DRUGBANK:DB09021""","""drug""","""Benzoctamine""","""DrugBank"""
"""drug_drug""","""synergistic interaction""","""DRUGBANK:DB04743""","""drug""","""Nimesulide""","""DrugBank""","""DRUGBANK:DB01291""","""drug""","""Pirbuterol""","""DrugBank"""
"""drug_drug""","""synergistic interaction""","""DRUGBANK:DB09190""","""drug""","""Talopram""","""DrugBank""","""DRUGBANK:DB13082""","""drug""","""Nefiracetam""","""DrugBank"""
"""drug_drug""","""synergistic interaction""","""DRUGBANK:DB08962""","""drug""","""Glibornuride""","""DrugBank""","""DRUGBANK:DB00620""","""drug""","""Triamcinolone""","""DrugBank"""
…,…,…,…,…,…,…,…,…,…
"""drug_drug""","""synergistic interaction""","""DRUGBANK:DB13491""","""drug""","""Fluperolone""","""DrugBank""","""DRUGBANK:DB00531""","""drug""","""Cyclophosphamide""","""DrugBank"""
"""drug_drug""","""synergistic interaction""","""DRUGBANK:DB00496""","""drug""","""Darifenacin""","""DrugBank""","""DRUGBANK:DB01065""","""drug""","""Melatonin""","""DrugBank"""
"""drug_drug""","""synergistic interaction""","""DRUGBANK:DB00718""","""drug""","""Adefovir dipivoxil""","""DrugBank""","""DRUGBANK:DB00713""","""drug""","""Oxacillin""","""DrugBank"""
